# User clustering

Segment users with at least five high ratings into taste personas. Each scaled column has one vote, so genres and production features dominate and rating habits are a side detail. Centers are learned on 80% of eligible users. Profiles are shown for 2 through 8 groups. Assignments are written only after `CHOSEN_K` is set.


## 1. Load and build the cluster matrix

One row per user with `n_high >= 5`. Taste columns in this file are already means over ratings of 4 or 5. `cast_film_count_log` is the mean of the three actor film-count logs, so a famous cast gets one vote.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT = Path("..")
PROCESSED = ROOT / "data" / "processed"
MIN_HIGH_RATINGS = 5
K_VALUES = range(2, 9)
RANDOM_STATE = 42
SILHOUETTE_SAMPLE_SIZE = 10_000

users = pd.read_csv(PROCESSED / "user_features.csv")
eligible = users.loc[users["n_high"] >= MIN_HIGH_RATINGS].reset_index(drop=True)

genre_cols = [c for c in eligible.columns if c.startswith("genre_") and c != "genre_entropy"]
eligible["log_n_ratings"] = np.log1p(eligible["n_ratings"])
eligible["cast_film_count_log"] = eligible[
    ["actor_1_film_count_log", "actor_2_film_count_log", "actor_3_film_count_log"]
].mean(axis=1)

cluster_cols = (
    ["log_n_ratings", "mean_rating", "user_rating_std"]
    + genre_cols
    + ["movie_age", "runtime", "budget_log", "director_film_count_log", "cast_film_count_log"]
)
left_out = [
    "n_high",
    "pct_high",
    "pct_low",
    "genre_entropy",
    "log_roi",
    "lang_en",
    "movie_rating_std",
    "mean_rating_vs_movie",
]

missing = [c for c in cluster_cols if c not in eligible.columns]
if missing:
    raise ValueError(f"Missing cluster columns: {missing}")
if len(genre_cols) != 19 or len(cluster_cols) != 27:
    raise ValueError(f"Expected 19 genres and 27 cluster columns, got {len(genre_cols)} and {len(cluster_cols)}")
overlap = sorted(set(cluster_cols) & set(left_out))
if overlap:
    raise ValueError(f"Left-out columns entered the matrix: {overlap}")
if eligible[cluster_cols].isna().any().any():
    raise ValueError("Cluster matrix has missing values")

print(f"Eligible users (n_high >= {MIN_HIGH_RATINGS}): {len(eligible):,}")
print(f"Cluster columns ({len(cluster_cols)}): {cluster_cols}")


Eligible users (n_high >= 5): 431,528
Cluster columns (27): ['log_n_ratings', 'mean_rating', 'user_rating_std', 'genre_action', 'genre_adventure', 'genre_animation', 'genre_comedy', 'genre_crime', 'genre_documentary', 'genre_drama', 'genre_family', 'genre_fantasy', 'genre_foreign', 'genre_history', 'genre_horror', 'genre_music', 'genre_mystery', 'genre_romance', 'genre_science_fiction', 'genre_thriller', 'genre_war', 'genre_western', 'movie_age', 'runtime', 'budget_log', 'director_film_count_log', 'cast_film_count_log']


## 2. Train/test split

80% of eligible users define the centers. The other 20% are placed into those groups after the fit.


In [2]:
train_index, test_index = train_test_split(
    eligible.index, test_size=0.2, random_state=RANDOM_STATE
)
eligible["split"] = "test"
eligible.loc[train_index, "split"] = "train"
is_train = eligible["split"].eq("train").to_numpy()

print(f"Train users: {int(is_train.sum()):,}")
print(f"Test users: {int((~is_train).sum()):,}")


Train users: 345,222
Test users: 86,306


## 3. Scale on the train users

Standardize each cluster column with the train mean and standard deviation. Test users use that same scaling.


In [3]:
scaler = StandardScaler()
x_train = scaler.fit_transform(eligible.loc[is_train, cluster_cols])
x_test = scaler.transform(eligible.loc[~is_train, cluster_cols])
x_scaled = np.empty((len(eligible), len(cluster_cols)), dtype=np.float64)
x_scaled[is_train] = x_train
x_scaled[~is_train] = x_test
print(f"Train matrix: {x_train.shape}")


Train matrix: (345222, 27)


## 4. Fit K-Means for k = 2 through 8

Centers are fit on the scaled train matrix only. Test users are assigned to the nearest train center.


In [4]:
labels_by_k = {}
for k in K_VALUES:
    model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    train_labels = model.fit_predict(x_train)
    test_labels = model.predict(x_test)
    labels = np.empty(len(eligible), dtype=np.int32)
    labels[is_train] = train_labels
    labels[~is_train] = test_labels
    labels_by_k[k] = labels
    print(f"k={k}  inertia={model.inertia_:,.0f}")


k=2  inertia=8,294,939


k=3  inertia=7,704,165


k=4  inertia=7,324,848


k=5  inertia=7,020,319


k=6  inertia=6,766,192


k=7  inertia=6,514,022


k=8  inertia=6,319,964


## 5. Persona profiles

For each k, sizes and the silhouette use train users. The silhouette is computed on one fixed sample of 10,000 train users so the scores can be compared. Standardized means are gaps from the train average, in standard deviations. `n_ratings` is shown beside `movie_age`, western, horror, and music because those taste columns rise with how many movies someone rates.


In [5]:
# Same train users for every k, so silhouette differences come from the grouping.
sample_pool = np.flatnonzero(is_train)
silhouette_sample = np.random.default_rng(RANDOM_STATE).choice(
    sample_pool, size=SILHOUETTE_SAMPLE_SIZE, replace=False
)

train = eligible.loc[is_train].copy()
profile_value_cols = [
    "budget_log",
    "movie_age",
    "runtime",
    "director_film_count_log",
    "cast_film_count_log",
    "n_ratings",
    "mean_rating",
    "user_rating_std",
]
activity_cols = ["n_ratings", "movie_age", "genre_western", "genre_horror", "genre_music"]


def top_genre_labels(genre_means, n=3):
    top = genre_means.nlargest(n)
    return ", ".join(f"{name.removeprefix('genre_')} {value:.2f}" for name, value in top.items())


summary_rows = []
details = {}
for k in K_VALUES:
    labels = labels_by_k[k]
    train_labels = labels[is_train]
    silhouette = silhouette_score(x_scaled[silhouette_sample], labels[silhouette_sample])
    counts = pd.Series(train_labels).value_counts().sort_index()
    share = counts / counts.sum()
    summary_rows.append(
        {
            "k": k,
            "silhouette": silhouette,
            "smallest_n": int(counts.min()),
            "smallest_share": float(share.min()),
        }
    )

    persona = train.copy()
    persona["persona_id"] = train_labels
    genre_means = persona.groupby("persona_id")[genre_cols].mean()
    profile = persona.groupby("persona_id")[profile_value_cols].mean()
    profile.insert(0, "n_users", counts)
    profile.insert(1, "share", share.to_numpy())
    profile.insert(2, "top_genres", genre_means.apply(top_genre_labels, axis=1))

    scaled_train = pd.DataFrame(x_train, columns=cluster_cols)
    scaled_train["persona_id"] = train_labels
    standardized = scaled_train.groupby("persona_id")[cluster_cols].mean().T
    standardized.columns = [f"persona_{int(col)}" for col in standardized.columns]

    activity = persona.groupby("persona_id")[activity_cols].mean()
    activity.insert(0, "n_users", counts)
    details[k] = (profile, standardized, activity)

summary = pd.DataFrame(summary_rows).set_index("k")
print(f"Silhouette sample: {SILHOUETTE_SAMPLE_SIZE:,} train users")
display(summary.style.format({"silhouette": "{:.3f}", "smallest_share": "{:.1%}"}))

profile_format = {
    "n_users": "{:.0f}",
    "share": "{:.1%}",
    "budget_log": "{:.2f}",
    "movie_age": "{:.2f}",
    "runtime": "{:.1f}",
    "director_film_count_log": "{:.2f}",
    "cast_film_count_log": "{:.2f}",
    "n_ratings": "{:.1f}",
    "mean_rating": "{:.3f}",
    "user_rating_std": "{:.3f}",
}
activity_format = {
    "n_users": "{:.0f}",
    "n_ratings": "{:.1f}",
    "movie_age": "{:.2f}",
    "genre_western": "{:.3f}",
    "genre_horror": "{:.3f}",
    "genre_music": "{:.3f}",
}
for k in K_VALUES:
    profile, standardized, activity = details[k]
    print(f"\nk = {k}")
    display(profile.style.format(profile_format))
    display(standardized.style.format("{:.2f}"))
    display(activity.style.format(activity_format))


Silhouette sample: 10,000 train users


,silhouette,smallest_n,smallest_share
k,,,
2,0.109,160520,46.5%
3,0.102,109431,31.7%
4,0.086,69631,20.2%
5,0.086,18203,5.3%
6,0.088,17060,4.9%
7,0.087,14,0.0%
8,0.084,14,0.0%



k = 2


,n_users,share,top_genres,budget_log,movie_age,runtime,director_film_count_log,cast_film_count_log,n_ratings,mean_rating,user_rating_std
persona_id,,,,,,,,,,,
0,184702,53.5%,"drama 0.56, comedy 0.40, romance 0.25",16.91,29.57,119.0,1.35,1.97,123.9,3.608,1.013
1,160520,46.5%,"drama 0.46, action 0.43, thriller 0.38",17.51,28.34,122.3,1.47,2.13,112.5,3.762,0.940


,persona_0,persona_1
log_n_ratings,0.04,-0.05
mean_rating,-0.16,0.18
user_rating_std,0.14,-0.16
genre_action,-0.63,0.73
genre_adventure,-0.42,0.48
genre_animation,0.12,-0.14
genre_comedy,0.39,-0.44
genre_crime,-0.29,0.33
genre_documentary,0.22,-0.26
genre_drama,0.27,-0.32


...

,n_users,n_ratings,movie_age,genre_western,genre_horror,genre_music
persona_id,,,,,,
0,53180,46.1,27.49,0.005,0.024,0.007
1,63671,69.3,26.43,0.005,0.018,0.024
2,66192,127.5,32.44,0.016,0.031,0.038
3,17318,53.1,27.02,0.015,0.021,0.024
4,14,25.1,26.98,0.020,0.023,0.032
5,41335,76.1,29.56,0.018,0.015,0.018
6,103512,208.5,29.24,0.017,0.060,0.017



k = 8


,n_users,share,top_genres,budget_log,movie_age,runtime,director_film_count_log,cast_film_count_log,n_ratings,mean_rating,user_rating_std
persona_id,,,,,,,,,,,
0,36503,10.6%,"drama 0.64, action 0.32, thriller 0.26",17.31,30.07,130.1,1.49,2.09,78.9,3.572,0.983
1,15996,4.6%,"comedy 0.47, drama 0.35, adventure 0.32",17.45,26.99,113.5,1.26,1.95,51.3,3.746,0.972
2,59666,17.3%,"drama 0.67, comedy 0.31, crime 0.21",16.38,32.46,121.5,1.35,1.88,122.1,3.490,1.063
3,41330,12.0%,"drama 0.57, thriller 0.46, action 0.35",17.48,26.92,121.6,1.42,2.16,55.1,3.720,0.972
4,14,0.0%,"drama 0.60, comedy 0.32, romance 0.29",17.02,26.98,127.4,1.19,1.80,25.1,3.733,0.963
5,55197,16.0%,"comedy 0.53, drama 0.50, romance 0.36",17.25,26.44,116.0,1.32,2.03,65.5,3.720,0.996
6,99373,28.8%,"drama 0.44, comedy 0.34, action 0.34",17.21,29.43,118.4,1.43,2.03,222.8,3.719,0.938
7,37143,10.8%,"action 0.60, thriller 0.44, drama 0.37",17.78,28.16,124.1,1.53,2.19,51.7,3.853,0.934


,persona_0,persona_1,persona_2,persona_3,persona_4,persona_5,persona_6,persona_7
log_n_ratings,-0.29,-0.66,0.08,-0.52,-1.10,-0.43,0.83,-0.57
mean_rating,-0.24,0.15,-0.42,0.09,0.12,0.09,0.09,0.39
user_rating_std,0.02,-0.03,0.35,-0.03,-0.07,0.07,-0.17,-0.19
genre_action,0.05,-0.08,-0.94,0.24,-0.49,-0.66,0.18,1.70
genre_adventure,-0.12,1.04,-0.71,-0.38,0.17,-0.64,0.29,1.42
genre_animation,-0.26,2.76,-0.20,-0.38,1.06,-0.07,0.07,-0.26
genre_comedy,-0.72,0.89,-0.17,-0.61,-0.09,1.24,0.02,-0.63
genre_crime,-0.16,-0.98,-0.00,1.25,-0.60,-0.54,-0.12,0.32
genre_documentary,-0.18,-0.06,0.85,-0.21,-0.01,-0.11,-0.17,-0.32
genre_drama,0.72,-1.00,0.95,0.31,0.49,-0.11,-0.42,-0.87


,n_users,n_ratings,movie_age,genre_western,genre_horror,genre_music
persona_id,,,,,,
0,36503,78.9,30.07,0.020,0.015,0.020
1,15996,51.3,26.99,0.015,0.021,0.024
2,59666,122.1,32.46,0.016,0.029,0.039
3,41330,55.1,26.92,0.006,0.036,0.012
4,14,25.1,26.98,0.020,0.023,0.032
5,55197,65.5,26.44,0.006,0.016,0.024
6,99373,222.8,29.43,0.017,0.059,0.019
7,37143,51.7,28.16,0.009,0.021,0.005


## 6. Save assignments

Set `CHOSEN_K` to an integer from 2 through 8 to write `data/processed/user_personas.csv` with `customer_id`, `persona_id`, and `split`. Leave it as `None` to skip the save. Persona names belong in this notebook after that choice.


In [6]:
CHOSEN_K = None

if CHOSEN_K is None:
    print("CHOSEN_K is unset. Skipping save.")
else:
    if CHOSEN_K not in K_VALUES:
        raise ValueError("CHOSEN_K must be an integer from 2 through 8")
    personas = eligible[["customer_id", "split"]].copy()
    personas["persona_id"] = labels_by_k[CHOSEN_K]
    personas = personas[["customer_id", "persona_id", "split"]]
    out_path = PROCESSED / "user_personas.csv"
    personas.to_csv(out_path, index=False)
    print(f"Saved {out_path.name}: {len(personas):,} rows")


CHOSEN_K is unset. Skipping save.
